In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Inicializa o Spark para puxar a amostra da camada Gold
spark = (
    SparkSession.builder.appName("TCC_Modelo_Exploratorio")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print(
    "📥 Lendo a fato_bacen da camada Gold e extraindo uma amostra representativa..."
)
df_fato_spark = spark.read.parquet("../data/processed/gold/fato_bacen")

# Extraindo 5% da base para teste inicial rápido e seguro na memória local
# (Garante velocidade sem estourar a RAM do seu PC)
df_sample_spark = df_fato_spark.sample(fraction=0.01, seed=42)
df_pd = df_sample_spark.toPandas()

print(f"📊 Amostra carregada no Pandas com {len(df_pd):,} linhas.")

# 2. Definindo a Variável Alvo (Target) e as Features (Atributos)
# O target é o nosso id_perfil_operacao (0 = Adimplente, 1 = Inadimplente)
target = "id_perfil_operacao"

# Selecionando colunas numéricas de saldo e IDs categóricos para teste
features_numericas = [
    "a_vencer_ate_90_dias",
    "a_vencer_de_91_ate_360_dias",
    "carteira_ativa",
    "numero_de_operacoes",
]
features_categoricas = [
    "id_uf",
    "id_segmento",
    "id_cliente",
    "id_porte",
    "id_modalidade",
    "id_indexador",
]

X = df_pd[features_numericas + features_categoricas]
y = df_pd[target]

# Tratando eventuais nulos nas features numéricas
X[features_numericas] = X[features_numericas].fillna(0)
X[features_categoricas] = X[features_categoricas].fillna(-1).astype(str)

# Divisão de Treino e Teste (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Construindo Pipeline de Pré-processamento e Modelo (Regressão Logística - GLM baseline)
print("\n⚙️ Treinando Modelo Estatístico (Regressão Logística / GLM)...")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_numericas),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            features_categoricas,
        ),
    ]
)

# Usando Regressão Logística para avaliarmos os coeficientes diretos (como pedido na ata)
model = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression(max_iter=500, class_weight="balanced"))]
)

model.fit(X_train, y_train)

# 4. Avaliação com Matriz de Confusão e Métricas
y_pred = model.predict(X_test)

print("\n=== MATRIZ DE CONFUSÃO ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

# 5. Analisando quais colunas mais "conversam" com a inadimplência (Coeficientes)
print("\n=== TOP FATORES QUE MAIS Puxam PARA INADIMPLÊNCIA (COEFICIENTES) ===")
# Extraindo nomes das features após o OneHotEncoder
cat_encoder = model.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cat_features = cat_encoder.get_feature_names_out(features_categoricas)
all_features = list(features_numericas) + list(encoded_cat_features)

coefficients = model.named_steps["classifier"].coef_[0]
df_coef = pd.DataFrame({"Feature": all_features, "Coeficiente": coefficients})
df_coef = df_coef.sort_values(by="Coeficiente", ascending=False)

print("--- Maiores Pesos Positivos (Empurram para Inadimplência = 1) ---")
print(df_coef.head(10))

print("\n--- Maiores Pesos Negativos (Empurram para Adimplência = 0) ---")
print(df_coef.tail(10))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/09 12:24:18 WARN Utils: Your hostname, DESKTOP-P0CSUOC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/09 12:24:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/09 12:24:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/09 12:24:25 WARN Utils: Service 'Spar